<a href="https://colab.research.google.com/github/eunyeongkimm/multimodal_user_needs_understanding/blob/main/qwen3_omni_test_vllm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===== 셀 1 (교체): vLLM 재설치 =====
!pip uninstall -y -q vllm vllm-omni
!pip install -q vllm qwen-omni-utils soundfile librosa
import vllm; print("vllm", vllm.__version__)

vllm 0.26.0


In [5]:
# ===== 셀 2 : 구글드라이브 마운트 =====
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, os
AUDIO_ROOT = '/content/drive/MyDrive/audio_seg'
PARQUET_DIR = '/content/drive/MyDrive/audio_seg_2'

manifest = pd.read_parquet(os.path.join(PARQUET_DIR, 'audio_seg_manifest.parquet'))
gold = pd.read_parquet(os.path.join(PARQUET_DIR, 'gold_actual_batch1_final.parquet'))
gold_map = dict(zip(gold.call_id, gold.label))
print("manifest:", manifest.shape, "| gold:", gold.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
manifest: (1119, 11) | gold: (19847, 5)


In [1]:
import torch
print("free:", round(torch.cuda.mem_get_info()[0]/1e9,1), "GB / total:", round(torch.cuda.mem_get_info()[1]/1e9,1), "GB")

free: 42.0 GB / total: 42.4 GB


In [2]:
# ===== vLLM 로드: Colab 안전 버전 =====

import os
import sys

os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

# DEBUG 쓰지 않음
os.environ["VLLM_LOGGING_LEVEL"] = "WARNING"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# progress bar도 최대한 억제
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

MODEL_ID = "cyankiwi/Qwen3-Omni-30B-A3B-Thinking-AWQ-4bit"

# 원래 Colab 출력 보관
_original_stdout = sys.stdout
_original_stderr = sys.stderr

# 실제 파일 객체 → fileno() 지원
_vllm_log = open("/tmp/vllm_load.log", "w")

# ★ vLLM import 전에 변경
sys.stdout = _vllm_log
sys.stderr = _vllm_log

load_error = None

try:
    from vllm import LLM, SamplingParams

    llm = LLM(
        model=MODEL_ID,
        runner="generate",
        trust_remote_code=True,
        tensor_parallel_size=1,

        gpu_memory_utilization=0.90,
        max_model_len=4096,
        max_num_seqs=1,

        limit_mm_per_prompt={
            "audio": 5,
            "image": 1,
            "video": 0,
        },

        enforce_eager=True,
        generation_config="vllm",
    )

except Exception as e:
    load_error = e

finally:
    # Colab 화면 출력 복구
    sys.stdout = _original_stdout
    sys.stderr = _original_stderr

    # 일부 logger가 이 파일을 계속 참조할 수 있으므로
    # 여기서는 일부러 close하지 않음
    _vllm_log.flush()


if load_error is None:
    print("✅ vLLM loaded OK")

else:
    print("❌ vLLM load failed:")
    print(repr(load_error))

    # 전체 로그 절대 출력하지 않고 마지막 40줄만
    with open("/tmp/vllm_load.log", "r", errors="replace") as f:
        lines = f.readlines()

    print("\n===== 마지막 로그 40줄 =====")
    print("".join(lines[-40:]))

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


✅ vLLM loaded OK


In [6]:
# 이 모델의 실제 오디오 토큰 확인
from transformers import AutoProcessor
proc = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
tok = proc.tokenizer if hasattr(proc, "tokenizer") else proc
# 오디오 관련 스페셜 토큰 찾기
specials = tok.all_special_tokens if hasattr(tok, "all_special_tokens") else []
print("special tokens:", specials)
for t in (tok.additional_special_tokens if hasattr(tok, "additional_special_tokens") else []):
    if "audio" in t.lower() or "AUDIO" in t:
        print("audio token:", t)

special tokens: ['<|im_end|>', '<|endoftext|>', '<|audio_start|>', '<|audio_end|>', '<|audio_pad|>', '<|image_pad|>', '<|video_pad|>', '<|vision_start|>', '<|vision_end|>']


In [3]:
import time
from vllm import SamplingParams

sampling_test = SamplingParams(
    temperature=0.0,
    max_tokens=1,
)

print("text-only 시작")

t0 = time.time()

out = llm.generate(
    ["안녕하세요. A라고 답하세요."],
    sampling_params=sampling_test,
    use_tqdm=False,
)

elapsed = time.time() - t0

print(f"걸린 시간: {elapsed:.2f}초")
print("출력:", out[0].outputs[0].text)
print("생성 토큰:", len(out[0].outputs[0].token_ids))

text-only 시작
걸린 시간: 0.17초
출력:  


생성 토큰: 1


In [8]:
# ===== 헬퍼 함수 복구 셀 =====

import os
import re

CATEGORIES = [
    "환불요청",
    "주문취소",
    "불만제기",
    "배송확인",
    "교환반품",
    "구매진행",
    "서비스이용",
]


def get_call_wavs(call_id, max_utterances=5):
    rows = (
        manifest[manifest.call_id == call_id]
        .sort_values("utt_idx")
        .head(max_utterances)
    )

    paths = []

    for _, r in rows.iterrows():
        p = os.path.join(
            AUDIO_ROOT,
            call_id,
            os.path.basename(r.wav_path)
        )

        if os.path.exists(p):
            paths.append(p)

    return paths


def build_prompt(n_audio):
    audio_tags = "".join(
        f"발화{i}: "
        "<|audio_start|><|audio_pad|><|audio_end|>\n"
        for i in range(1, n_audio + 1)
    )

    cats = ", ".join(CATEGORIES)

    return (
        "<|im_start|>system\n"
        "당신은 한국어 콜센터 통화 분류기입니다. "
        "고객의 초반 발화 음성을 듣고 의도를 분류합니다."
        "<|im_end|>\n"

        "<|im_start|>user\n"
        f"{audio_tags}\n"

        f"위 발화들을 종합해 고객 의도를 "
        f"다음 7개 중 하나로 분류하세요: {cats}\n"

        "최종 답변은 반드시 "
        "'정답: <카테고리>' 형식으로 작성하세요."
        "<|im_end|>\n"

        "<|im_start|>assistant\n"
    )


def parse_pred(resp):
    m = re.search(
        r"정답\s*[:：]\s*([가-힣]+)",
        resp
    )

    if m:
        pred = m.group(1).strip()

        if pred in CATEGORIES:
            return pred

    tail = resp[-200:]

    for c in CATEGORIES:
        if c in tail:
            return c

    for c in CATEGORIES:
        if c in resp:
            return c

    return None


print("✅ helper functions loaded")

✅ helper functions loaded


In [9]:
# ===== 다음 진단: audio 1개 → audio 5개 =====

import time
import librosa
from vllm import SamplingParams

sampling_test = SamplingParams(
    temperature=0.0,
    max_tokens=1,
)

cid = manifest.call_id.unique()[0]
wavs = get_call_wavs(cid, max_utterances=5)

audios = []

for w in wavs:
    arr, _ = librosa.load(
        w,
        sr=16000,
        mono=True
    )
    audios.append((arr, 16000))


# --------------------------------------------
# TEST 1: audio 1개
# --------------------------------------------

print("audio 1개 시작")

t0 = time.time()

out = llm.generate(
    [{
        "prompt": build_prompt(1),
        "multi_modal_data": {
            "audio": [audios[0]]
        }
    }],
    sampling_params=sampling_test,
    use_tqdm=False,
)

print(f"audio 1개: {time.time() - t0:.2f}초")


# --------------------------------------------
# TEST 2: audio 5개
# --------------------------------------------

print("\naudio 5개 시작")

t0 = time.time()

out = llm.generate(
    [{
        "prompt": build_prompt(5),
        "multi_modal_data": {
            "audio": audios
        }
    }],
    sampling_params=sampling_test,
    use_tqdm=False,
)

print(f"audio 5개: {time.time() - t0:.2f}초")

audio 1개 시작
audio 1개: 0.34초

audio 5개 시작
audio 5개: 0.15초


In [14]:
# ===== 셀 5: 실제 1콜 추론 - max_tokens=512 버전 =====

import time
import re
import librosa
from vllm import SamplingParams


# --------------------------------------------------
# 1. 카테고리
# --------------------------------------------------

CATEGORIES = [
    "환불요청",
    "주문취소",
    "불만제기",
    "배송확인",
    "교환반품",
    "구매진행",
    "서비스이용",
]


# --------------------------------------------------
# 2. Sampling
# --------------------------------------------------
# Thinking 모델이 256토큰에서도 잘렸으므로 512로 증가

sampling = SamplingParams(
    temperature=0.0,
    max_tokens=512,
)


# --------------------------------------------------
# 3. 엄격한 결과 파싱
# --------------------------------------------------
# <think> 안에 등장한 카테고리는 예측으로 인정하지 않음.
# 반드시 "정답: 카테고리" 형태가 있어야 pred로 인정.

def parse_pred(resp):

    pattern = (
        r"정답\s*[:：]\s*"
        r"(환불요청|주문취소|불만제기|배송확인|"
        r"교환반품|구매진행|서비스이용)"
    )

    m = re.search(pattern, resp)

    if m:
        return m.group(1)

    return None


# --------------------------------------------------
# 4. 한 콜 추론 함수
# --------------------------------------------------

def predict_call(call_id):

    # 초반 고객 발화 최대 5개
    wavs = get_call_wavs(
        call_id,
        max_utterances=5
    )

    if not wavs:
        return {
            "call_id": call_id,
            "pred": None,
            "raw": "no_audio",
            "elapsed_sec": 0,
            "generated_tokens": 0,
            "finish_reason": None,
            "n_audio": 0,
            "audio_duration_sec": 0,
        }


    # ----------------------------------------------
    # Audio load
    # ----------------------------------------------

    audios = []
    durations = []

    for w in wavs:

        arr, _ = librosa.load(
            w,
            sr=16000,
            mono=True,
        )

        audios.append(
            (arr, 16000)
        )

        durations.append(
            len(arr) / 16000
        )


    # ----------------------------------------------
    # Prompt
    # ----------------------------------------------

    prompt = build_prompt(
        len(audios)
    )


    # ----------------------------------------------
    # Inference
    # ----------------------------------------------

    start = time.time()

    outputs = llm.generate(
        [
            {
                "prompt": prompt,

                "multi_modal_data": {
                    "audio": audios
                },
            }
        ],

        sampling_params=sampling,

        # progress bar 제거
        use_tqdm=False,
    )

    elapsed = time.time() - start


    # ----------------------------------------------
    # Result
    # ----------------------------------------------

    result = outputs[0].outputs[0]

    raw = result.text
    pred = parse_pred(raw)


    return {
        "call_id": call_id,

        "pred": pred,

        "raw": raw,

        "elapsed_sec": elapsed,

        "generated_tokens": len(
            result.token_ids
        ),

        "finish_reason": (
            result.finish_reason
        ),

        "n_audio": len(audios),

        "audio_duration_sec": sum(
            durations
        ),
    }


# ==================================================
# 5. 1콜 테스트
# ==================================================

cid = "J16_S000434"

result = predict_call(cid)


# --------------------------------------------------
# Gold label
# --------------------------------------------------

if "gold" in globals():

    gold_map = dict(
        zip(
            gold.call_id,
            gold.label
        )
    )

    gold_label = gold_map.get(cid)

else:
    gold_label = None


# --------------------------------------------------
# 결과 출력
# --------------------------------------------------

print("========== RESULT ==========")

print(
    "call_id :",
    result["call_id"]
)

print(
    "gold    :",
    gold_label
)

print(
    "pred    :",
    result["pred"]
)

print(
    "time    :",
    round(
        result["elapsed_sec"],
        2
    ),
    "sec"
)

print(
    "tokens  :",
    result["generated_tokens"]
)

print(
    "finish  :",
    result["finish_reason"]
)

print(
    "audio   :",
    result["n_audio"],
    "files /",
    round(
        result["audio_duration_sec"],
        1
    ),
    "sec"
)

print()

print("========== RAW ==========")

print(
    result["raw"]
)

========== RESULT ==========
call_id : J16_S000434
gold    : 서비스이용
pred    : None
time    : 46.15 sec
tokens  : 512
finish  : length
audio   : 5 files / 13.7 sec

========== RAW ==========
<think>
Okay, let's tackle this problem. So, I need to classify the customer's intent based on the given utterances. The options are: 환불요청 (refund request), 주문취소 (order cancellation), 불만제기 (complaint), 배송확인 (delivery confirmation), 교환반품 (exchange/refund), 구매진행 (purchase process), 서비스이용 (service usage).

First, let's look at each utterance one by one.

Utterance 1: "여보세요. 아까 뭐요?" translates to "Hello. What was that earlier?" Hmm, the customer is probably asking about something that happened before, maybe a previous interaction. But it's a bit vague.

Utterance 2: "엠베스 쓸 수 있어?" which is "Can I use Embees?" Maybe they're asking about a service or product called Embees. But the context isn't clear yet.

Utterance 3: "엠보트는 언제 올 수 있나요?" translates to "When can Embot come?" Maybe they're asking about delive

In [17]:
# ===== 셀 5: 7개 카테고리 강제 선택 추론 =====

import time
import librosa

from vllm import SamplingParams
from vllm.sampling_params import StructuredOutputsParams


# --------------------------------------------------
# 1. 카테고리
# --------------------------------------------------

CATEGORIES = [
    "환불요청",
    "주문취소",
    "불만제기",
    "배송확인",
    "교환반품",
    "구매진행",
    "서비스이용",
]


# --------------------------------------------------
# 2. Structured Output
# --------------------------------------------------
# 모델이 아래 7개 문자열 이외에는 생성할 수 없게 제한

structured = StructuredOutputsParams(
    choice=CATEGORIES
)

sampling = SamplingParams(
    temperature=0.0,

    # 카테고리명만 생성하므로 충분
    max_tokens=16,

    structured_outputs=structured,
)


# --------------------------------------------------
# 3. 분류용 prompt
# --------------------------------------------------

def build_classification_prompt(n_audio):

    audio_tags = "".join(
        f"발화{i}: "
        "<|audio_start|>"
        "<|audio_pad|>"
        "<|audio_end|>\n"
        for i in range(1, n_audio + 1)
    )

    categories = ", ".join(CATEGORIES)

    return (
        "<|im_start|>system\n"
        "당신은 한국어 콜센터 고객 의도 분류기입니다.\n"
        "고객의 음성 발화들을 듣고 전체 통화 의도를 분류하세요.\n"
        "설명하지 말고 가장 적합한 카테고리 하나만 선택하세요.\n"
        "<|im_end|>\n"

        "<|im_start|>user\n"
        f"{audio_tags}\n"
        f"가능한 카테고리: {categories}\n"
        "위 발화들을 종합하여 가장 적합한 카테고리 하나를 선택하세요.\n"
        "<|im_end|>\n"

        "<|im_start|>assistant\n"
    )


# --------------------------------------------------
# 4. 한 콜 추론
# --------------------------------------------------

def predict_call(call_id):

    wavs = get_call_wavs(
        call_id,
        max_utterances=5
    )

    if not wavs:
        return {
            "call_id": call_id,
            "pred": None,
            "raw": "no_audio",
            "elapsed_sec": 0,
            "generated_tokens": 0,
            "finish_reason": None,
            "n_audio": 0,
            "audio_duration_sec": 0,
        }


    # Audio load
    audios = []
    durations = []

    for w in wavs:

        arr, _ = librosa.load(
            w,
            sr=16000,
            mono=True,
        )

        audios.append((arr, 16000))
        durations.append(len(arr) / 16000)


    prompt = build_classification_prompt(
        len(audios)
    )


    # Inference
    start = time.time()

    outputs = llm.generate(
        [
            {
                "prompt": prompt,
                "multi_modal_data": {
                    "audio": audios
                },
            }
        ],
        sampling_params=sampling,
        use_tqdm=False,
    )

    elapsed = time.time() - start


    result = outputs[0].outputs[0]

    raw = result.text.strip()

    # structured choice이므로 그대로 prediction
    pred = raw if raw in CATEGORIES else None


    return {
        "call_id": call_id,
        "pred": pred,
        "raw": raw,
        "elapsed_sec": elapsed,
        "generated_tokens": len(result.token_ids),
        "finish_reason": result.finish_reason,
        "n_audio": len(audios),
        "audio_duration_sec": sum(durations),
    }


# ==================================================
# 5. 동일한 1콜 테스트
# ==================================================

cid = "J16_S000434"

result = predict_call(cid)


gold_map = dict(
    zip(
        gold.call_id,
        gold.label
    )
)

gold_label = gold_map.get(cid)


print("========== RESULT ==========")
print("call_id :", result["call_id"])
print("gold    :", gold_label)
print("pred    :", result["pred"])
print("time    :", round(result["elapsed_sec"], 2), "sec")
print("tokens  :", result["generated_tokens"])
print("finish  :", result["finish_reason"])
print(
    "audio   :",
    result["n_audio"],
    "files /",
    round(result["audio_duration_sec"], 1),
    "sec"
)

print()
print("========== RAW ==========")
print(result["raw"])

========== RESULT ==========
call_id : J16_S000434
gold    : 서비스이용
pred    : 불만제기
time    : 1.48 sec
tokens  : 6
finish  : stop
audio   : 5 files / 13.7 sec

========== RAW ==========
불만제기


In [18]:
# ===== 셀 6: 250 전체 추론 + 저장 =====
import pandas as pd, time

call_ids = list(manifest.call_id.unique())
rows = []
t_start = time.time()
for i, cid in enumerate(call_ids):
    r = predict_call(cid)
    r["gold"] = gold_map.get(cid)
    rows.append(r)
    if (i+1) % 25 == 0:
        print(f"{i+1}/250  ({time.time()-t_start:.0f}s)")

df = pd.DataFrame(rows)
save_path = "/content/drive/MyDrive/audio_seg_2/qwen3omni_30b_predictions.parquet"
df.to_parquet(save_path)
print("saved:", save_path)

# 진단
parsed = df.dropna(subset=["pred"])
print(f"\n파싱 성공: {len(parsed)}/250, 실패: {df.pred.isna().sum()}")
print(f"accuracy: {(parsed.pred==parsed.gold).mean():.3f}")
print(f"평균 추론시간: {df.elapsed_sec.mean():.2f}s/콜")
print("\n예측 분포:\n", df.pred.value_counts())
print("\ngold 분포:\n", df.gold.value_counts())

25/250  (64s)
50/250  (124s)
75/250  (185s)
100/250  (244s)
125/250  (304s)
150/250  (366s)
175/250  (423s)
200/250  (480s)
225/250  (536s)
250/250  (597s)
saved: /content/drive/MyDrive/audio_seg_2/qwen3omni_30b_predictions.parquet

파싱 성공: 250/250, 실패: 0
accuracy: 0.328
평균 추론시간: 0.69s/콜

예측 분포:
 pred
불만제기     101
배송확인      46
환불요청      34
교환반품      25
주문취소      24
구매진행      16
서비스이용      4
Name: count, dtype: int64

gold 분포:
 gold
환불요청     87
서비스이용    51
불만제기     50
배송확인     24
교환반품     19
구매진행     11
주문취소      8
Name: count, dtype: int64
